# Trading Data Cleaning & Feature Engineering
**Internship task — synthetic BTCUSDT trades and order-book snapshots**

This notebook follows the supplied EDA checklist and task README. It covers data preparation and feature engineering, not model fine-tuning. The data is synthetic, not real market data or financial advice.

**Run:** Keep `trades_orderbook_STUDENT_dirty.csv` beside this notebook, then run cells from top to bottom. Outputs are written to `outputs/`.

## 1. Imports and configuration

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    import seaborn as sns
    HAS_SEABORN = True
except ImportError:
    HAS_SEABORN = False

pd.set_option("display.max_columns", 50)
INPUT_CSV = Path("trades_orderbook_STUDENT_dirty.csv")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if not INPUT_CSV.exists():
    fallback = Path("/mnt/data/trades_orderbook_STUDENT_dirty.csv")
    if fallback.exists():
        INPUT_CSV = fallback
    else:
        raise FileNotFoundError("Put the dirty CSV beside this notebook or update INPUT_CSV.")
print("Input:", INPUT_CSV.resolve())

## 2. Load and inspect

In [ ]:
raw = pd.read_csv(INPUT_CSV)
df = raw.copy()
display(df.head())
print("Shape:", df.shape)
df.info()
display(df.dtypes.rename("dtype").to_frame())

## 3. Summaries

In [ ]:
display(df.describe(include="all").T)
for column in ["side", "symbol"]:
    if column in df:
        print(f"Value counts: {column}")
        display(df[column].value_counts(dropna=False).to_frame("count"))
for column in ["price", "quantity"]:
    if column in df:
        numeric = pd.to_numeric(df[column].astype("string").str.replace(r"[$,]", "", regex=True).str.strip(), errors="coerce")
        print(f"{column}: min={numeric.min()}, max={numeric.max()}, mean={numeric.mean()}, median={numeric.median()}")

## 4. Diagnose data-quality issues

In [ ]:
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))
print("Duplicate trade_id rows:", int(df["trade_id"].duplicated().sum()))
def timestamp_kind(value):
    if pd.isna(value): return "missing"
    s = str(value).strip()
    if not s: return "missing"
    if s.isdigit():
        if len(s) >= 13: return "Unix milliseconds (likely)"
        if len(s) >= 10: return "Unix seconds (likely)"
        return "numeric timestamp (unit unclear)"
    if "/" in s: return "slash-separated date"
    return "datetime/text"
display(df["timestamp"].map(timestamp_kind).value_counts(dropna=False).to_frame("count"))
audit = df.copy()
for col in ["price", "quantity", "bid_price_1", "ask_price_1", "bid_qty_1", "ask_qty_1"]:
    audit[col] = pd.to_numeric(audit[col].astype("string").str.replace(r"[$,]", "", regex=True).str.strip(), errors="coerce")
for col in ["price", "quantity"]:
    print(f"{col} missing/unparseable:", int(audit[col].isna().sum()), "| <= 0:", int((audit[col] <= 0).sum()))
print("Crossed books (bid_price_1 > ask_price_1):", int((audit["bid_price_1"] > audit["ask_price_1"]).sum()))

## 5. Clean the data — decisions are documented

Rules: keep the first row per `trade_id`; normalize side to `BUY`/`SELL`; normalize common symbol spellings to `BTCUSDT`; strip `$` and commas from numeric text; parse ISO dates, day-first slash dates, Unix seconds and milliseconds; drop rows missing essential fields; drop non-positive trade/book prices, non-positive trade quantity, negative order-book quantities, and crossed books. Zero book quantity is allowed. Finally, flag positive trade-price outliers using the Q1/Q3 IQR rule at 3×IQR and remove them. These are explicit assumptions, not the only possible cleaning policy.

In [ ]:
clean = raw.copy()
clean["_source_row"] = np.arange(len(clean))
clean["side"] = clean["side"].astype("string").str.strip().str.upper()
symbol = clean["symbol"].astype("string").str.strip().str.upper()
clean["symbol"] = symbol.str.replace(r"[\\s/_-]", "", regex=True)

numeric_cols = ["price", "quantity", "bid_price_1", "bid_qty_1", "bid_price_2", "bid_qty_2",
                "bid_price_3", "bid_qty_3", "ask_price_1", "ask_qty_1", "ask_price_2",
                "ask_qty_2", "ask_price_3", "ask_qty_3"]
for col in numeric_cols:
    clean[col] = pd.to_numeric(clean[col].astype("string").str.replace(r"[$,]", "", regex=True).str.strip(), errors="coerce")

def parse_mixed_timestamp(value):
    if pd.isna(value): return pd.NaT
    s = str(value).strip()
    if not s: return pd.NaT
    if s.isdigit():
        try:
            number = int(s)
            if len(s) >= 13 or number >= 100_000_000_000:
                return pd.to_datetime(number, unit="ms", errors="coerce", utc=True).tz_convert(None)
            if len(s) >= 10 or number >= 1_000_000_000:
                return pd.to_datetime(number, unit="s", errors="coerce", utc=True).tz_convert(None)
        except (ValueError, OverflowError):
            return pd.NaT
    if "/" in s:
        return pd.to_datetime(s, dayfirst=True, errors="coerce")
    return pd.to_datetime(s, errors="coerce")

clean["timestamp"] = pd.to_datetime(clean["timestamp"].map(parse_mixed_timestamp), errors="coerce")
cleaning_log = []
def apply_filter(frame, keep_mask, rule, reason):
    before = len(frame)
    result = frame.loc[keep_mask].copy()
    cleaning_log.append({"rule": rule, "rows_removed": int(before-len(result)), "reason": reason})
    print(f"{rule}: removed {before-len(result)} row(s)")
    return result

clean = apply_filter(clean, ~clean["trade_id"].duplicated(keep="first"),
                     "Duplicate trade_id", "Kept first occurrence of each trade_id.")
clean = apply_filter(clean, clean["side"].isin(["BUY", "SELL"]),
                     "Invalid side", "Only standardized BUY and SELL labels are retained.")
clean = apply_filter(clean, clean["symbol"].eq("BTCUSDT"),
                     "Missing/unsupported symbol", "Common BTCUSDT spellings normalized; unsupported symbols excluded.")
required = ["trade_id", "timestamp", "price", "quantity", "bid_price_1", "ask_price_1", "bid_qty_1", "ask_qty_1"]
clean = apply_filter(clean, clean[required].notna().all(axis=1),
                     "Missing/unparseable required fields", "Essential trade, time, and top-of-book fields are required.")
clean = apply_filter(clean, (clean["price"] > 0) & (clean["quantity"] > 0) &
                     (clean["bid_price_1"] > 0) & (clean["ask_price_1"] > 0),
                     "Non-positive price/quantity", "Prices and executed trade quantity must be positive.")
book_qty_cols = ["bid_qty_1", "bid_qty_2", "bid_qty_3", "ask_qty_1", "ask_qty_2", "ask_qty_3"]
clean = apply_filter(clean, (clean[book_qty_cols].isna() | (clean[book_qty_cols] >= 0)).all(axis=1),
                     "Negative book quantity", "Negative displayed order-book quantities are invalid; zero is allowed.")
clean = apply_filter(clean, clean["bid_price_1"] <= clean["ask_price_1"],
                     "Crossed top-of-book", "Dropped snapshots where best bid exceeds best ask.")
q1, q3 = clean["price"].quantile([0.25, 0.75])
iqr = q3 - q1
lower_bound, upper_bound = max(0, q1-3*iqr), q3+3*iqr
print(f"Trade-price IQR bounds (3 x IQR): {lower_bound:.4f} to {upper_bound:.4f}")
outliers = (clean["price"] < lower_bound) | (clean["price"] > upper_bound)
clean = apply_filter(clean, ~outliers, "Extreme trade-price outlier",
                     "Removed prices outside Q1 - 3xIQR / Q3 + 3xIQR.")
clean = clean.sort_values(["timestamp", "trade_id"]).drop(columns=["_source_row"]).reset_index(drop=True)
cleaning_log = pd.DataFrame(cleaning_log)
display(cleaning_log)
print("Raw rows:", len(raw), "| Clean rows:", len(clean), "| Removed:", len(raw)-len(clean))
display(clean.isna().sum().sort_values(ascending=False).to_frame("remaining_missing"))
display(clean.head())

## 6. Engineer features

In [ ]:
clean["spread"] = clean["ask_price_1"] - clean["bid_price_1"]
clean["mid_price"] = (clean["ask_price_1"] + clean["bid_price_1"]) / 2
denom = clean["bid_qty_1"] + clean["ask_qty_1"]
clean["order_imbalance"] = np.where(denom > 0, (clean["bid_qty_1"]-clean["ask_qty_1"])/denom, np.nan)
clean = clean.sort_values("timestamp").reset_index(drop=True)
clean["return_1"] = clean["price"].pct_change()
clean["rolling_volatility_20"] = clean["return_1"].rolling(20, min_periods=10).std()
clean["sma_10"] = clean["price"].rolling(10, min_periods=10).mean()
clean["sma_30"] = clean["price"].rolling(30, min_periods=30).mean()
display(clean[["timestamp","price","spread","mid_price","order_imbalance","return_1","rolling_volatility_20","sma_10","sma_30"]].head(12))

## 7. Resample ticks to 1-minute OHLCV

In [ ]:
ticks = clean.set_index("timestamp").sort_index()
candles = ticks.resample("1min").agg(
    open=("price","first"), high=("price","max"), low=("price","min"), close=("price","last"),
    volume=("quantity","sum"), trade_count=("trade_id","count"),
    mean_spread=("spread","mean"), mean_order_imbalance=("order_imbalance","mean"),
    mid_price=("mid_price","mean")
).dropna(subset=["open","high","low","close"]).reset_index()
candles["return_1m"] = candles["close"].pct_change()
candles["volatility_10m"] = candles["return_1m"].rolling(10, min_periods=5).std()
candles["sma_5m"] = candles["close"].rolling(5, min_periods=5).mean()
candles["sma_15m"] = candles["close"].rolling(15, min_periods=15).mean()
display(candles.head())
print("1-minute candles:", len(candles))

## 8. Visualizations

In [ ]:
fig, ax = plt.subplots(figsize=(12,4))
ax.plot(clean["timestamp"], clean["price"], linewidth=.8)
ax.set(title="Cleaned Trade Price Over Time", xlabel="Time", ylabel="Price")
fig.autofmt_xdate(); plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(8,4))
ax.hist(clean["quantity"].dropna(), bins=40)
ax.set(title="Trade Quantity Distribution", xlabel="Trade quantity", ylabel="Frequency")
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(8,4))
ax.hist(clean["spread"].dropna(), bins=40)
ax.set(title="Top-of-Book Spread Distribution", xlabel="Spread (ask_1 - bid_1)", ylabel="Frequency")
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(6,4))
clean["side"].value_counts().reindex(["BUY","SELL"], fill_value=0).plot(kind="bar", ax=ax)
ax.set(title="Buy vs Sell Trade Counts", xlabel="Side", ylabel="Count")
plt.tight_layout(); plt.show()

corr_cols = ["price","quantity","spread","mid_price","order_imbalance","return_1",
             "rolling_volatility_20","sma_10","sma_30"]
corr = clean[corr_cols].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(10,7))
if HAS_SEABORN:
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
else:
    im = ax.imshow(corr, vmin=-1, vmax=1, aspect="auto")
    ax.set_xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(corr.index)), corr.index)
    for i in range(len(corr.index)):
        for j in range(len(corr.columns)):
            ax.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center", fontsize=8)
    fig.colorbar(im, ax=ax)
ax.set_title("Feature Correlation Heatmap")
plt.tight_layout(); plt.show()

## 9. Target and chronological train/test split

The target is whether the close is higher **five 1-minute candles later**. Future close is used only to construct the label and is excluded from the feature list. The final five candles have no label and are omitted. The split is chronological (first 70% train, last 30% test), never random.

In [ ]:
model_data = candles.sort_values("timestamp").reset_index(drop=True).copy()
model_data["future_close_5m"] = model_data["close"].shift(-5)
model_data["target_up_5m"] = (model_data["future_close_5m"] > model_data["close"]).astype(float)
model_data.loc[model_data["future_close_5m"].isna(), "target_up_5m"] = np.nan
feature_cols = ["open","high","low","close","volume","trade_count","mean_spread",
                "mean_order_imbalance","mid_price","return_1m","volatility_10m","sma_5m","sma_15m"]
model_table = model_data.dropna(subset=["target_up_5m"]).replace([np.inf,-np.inf], np.nan).copy()
split_idx = int(len(model_table)*0.70)
train, test = model_table.iloc[:split_idx].copy(), model_table.iloc[split_idx:].copy()
print("Rows:", len(model_table), "| Train:", len(train), "| Test:", len(test))
if len(train) and len(test):
    print("Train time:", train.timestamp.min(), "to", train.timestamp.max())
    print("Test time:", test.timestamp.min(), "to", test.timestamp.max())
    assert train.timestamp.max() < test.timestamp.min(), "Split is not chronological"
display(model_table[["timestamp", *feature_cols, "target_up_5m"]].head())

### Optional scaling (fit on training data only)

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
X_train, X_test = train[feature_cols].copy(), test[feature_cols].copy()
y_train, y_test = train["target_up_5m"].astype(int), test["target_up_5m"].astype(int)
imputer = SimpleImputer(strategy="median")
X_train_i = imputer.fit_transform(X_train)
X_test_i = imputer.transform(X_test)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_i)
X_test_scaled = scaler.transform(X_test_i)
print("Prepared shapes:", X_train_scaled.shape, X_test_scaled.shape)
display(y_train.value_counts(normalize=True).rename("train proportion").to_frame())
display(y_test.value_counts(normalize=True).rename("test proportion").to_frame())

## 10. Save outputs

In [ ]:
clean_path = OUTPUT_DIR / "trades_orderbook_cleaned.csv"
candles_path = OUTPUT_DIR / "trades_1min_ohlcv.csv"
model_path = OUTPUT_DIR / "trades_1min_model_dataset.csv"
log_path = OUTPUT_DIR / "cleaning_decisions_log.csv"
clean.to_csv(clean_path, index=False)
candles.to_csv(candles_path, index=False)
model_data.to_csv(model_path, index=False)
cleaning_log.to_csv(log_path, index=False)
for path in [clean_path, candles_path, model_path, log_path]:
    print(f"Saved {path} ({path.stat().st_size:,} bytes)")

## 11. Submission checklist

- [ ] Inspected row count, shape, columns, dtypes, and summaries
- [ ] Reported missing values, duplicates, timestamp formats, non-positive values, and crossed books
- [ ] Documented each cleaning rule and removed-row count
- [ ] Removed duplicate `trade_id` values and sorted by time
- [ ] Added spread, mid-price, order imbalance, returns, rolling volatility, and moving averages
- [ ] Built 1-minute OHLCV candles
- [ ] Produced price, volume, spread, buy/sell, and correlation plots
- [ ] Created a next-five-minute direction target and time-based split
- [ ] Saved cleaned data and derived CSVs

Review the cleaning assumptions and outputs before submitting. This notebook is educational material using synthetic data, not financial advice.